In [7]:
# =====================================
# ENRIQUECIMIENTO CUALITATIVO CON LLM (Gemini)
# =====================================

import os
import json
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Cargar la API key desde el fichero .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    print("❌ No se encontró GEMINI_API_KEY en el fichero .env")
else:
    print(f"✅ API key cargada (termina en ...{API_KEY[-4:]})")
    client = genai.Client(api_key=API_KEY)

# Rutas
GOLD_DIR = Path("../data/gold")
PROCESSED_DIR = Path("../data/processed")

✅ API key cargada (termina en ...3isw)


In [6]:
from pathlib import Path

# ¿Existe el fichero .env?
env_path = Path("../.env")
print(f"Ruta buscada: {env_path.resolve()}")
print(f"¿Existe?: {env_path.exists()}")

if env_path.exists():
    with open(env_path) as f:
        contenido = f.read()
    print(f"\nNúmero de líneas: {len(contenido.splitlines())}")
    # Mostrar solo la estructura, no la clave
    for i, linea in enumerate(contenido.splitlines(), 1):
        if '=' in linea:
            nombre = linea.split('=')[0]
            valor = linea.split('=', 1)[1]
            print(f"  Línea {i}: variable '{nombre}' con valor de {len(valor)} caracteres")
        else:
            print(f"  Línea {i}: (sin '=') → '{linea[:30]}...'")

Ruta buscada: /Users/juana/Desktop/tfm-data-science-gtm/.env
¿Existe?: True

Número de líneas: 2
  Línea 1: (sin '=') → '...'
  Línea 2: (sin '=') → 'GEMINI_API_KEYAQ.Ab8RN6LA_eHK-...'


In [8]:
# =====================================
# PRUEBA DE CONEXIÓN CON GEMINI
# =====================================

try:
    respuesta = client.models.generate_content(
        model="gemini-2.0-flash",
        contents="Responde solo con la palabra: OK"
    )
    print("✅ Conexión funcionando")
    print(f"Respuesta: {respuesta.text}")
except Exception as e:
    print(f"❌ Error de conexión:")
    print(f"   {type(e).__name__}: {e}")

❌ Error de conexión:
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}


In [9]:
# =====================================
# PRUEBA DE CONEXIÓN CON GEMINI
# =====================================

MODELO = "gemini-3.6-flash"

try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents="Responde solo con la palabra: OK"
    )
    print("✅ Conexión funcionando")
    print(f"Modelo: {MODELO}")
    print(f"Respuesta: {respuesta.text}")
except Exception as e:
    print(f"❌ Error de conexión:")
    print(f"   {type(e).__name__}: {e}")

✅ Conexión funcionando
Modelo: gemini-3.6-flash
Respuesta: OK


In [10]:
# =====================================
# CARGAR CAPA GOLD Y TOMAR MUESTRA
# =====================================

gold = pd.read_parquet(GOLD_DIR / "gold_restaurantes_madrid.parquet")
print(f"Capa gold cargada: {len(gold)} restaurantes")

# Tomar una muestra de 5 restaurantes CON datos ricos para la prueba
# (que tengan cocina y barrio, para que el LLM tenga con qué trabajar)
muestra = gold[gold['cocina'].notna() & gold['barrio'].notna()].head(5)

print(f"\n=== MUESTRA DE PRUEBA ===")
print(muestra[['nombre', 'barrio', 'cocina', 'epigrafe_oficial', 'tiene_web']].to_string())

Capa gold cargada: 1688 restaurantes

=== MUESTRA DE PRUEBA ===
               nombre                barrio         cocina epigrafe_oficial  tiene_web
0  La Casa del Abuelo  SOL                        regional  BAR RESTAURANTE       True
1             Barinka  SOL                        peruvian      RESTAURANTE      False
2      Taberna Griega  UNIVERSIDAD                   greek      RESTAURANTE       True
5         Maricastaña  UNIVERSIDAD           international  BAR RESTAURANTE       True
6           La Prensa  UNIVERSIDAD                regional  BAR RESTAURANTE      False


In [11]:
# =====================================
# PROMPT Y FUNCIÓN DE ENRIQUECIMIENTO
# =====================================

def construir_prompt(restaurante):
    """Construye el prompt para un restaurante concreto."""
    nombre = restaurante['nombre']
    barrio = restaurante['barrio'].strip() if pd.notna(restaurante['barrio']) else "desconocido"
    cocina = restaurante['cocina'] if pd.notna(restaurante['cocina']) else "no especificada"
    epigrafe = restaurante['epigrafe_oficial'] if pd.notna(restaurante['epigrafe_oficial']) else "no especificado"
    tiene_web = "sí" if restaurante['tiene_web'] else "no"

    prompt = f"""Eres un analista de mercado especializado en el sector de restauración en Madrid.

A partir de la información disponible de un restaurante, infiere sus características comerciales. Basa tu inferencia en el nombre, el tipo de cocina y el barrio. No inventes datos factuales concretos (teléfonos, premios, fechas); limítate a caracterizar el perfil comercial.

DATOS DEL RESTAURANTE:
- Nombre: {nombre}
- Barrio (distrito Centro de Madrid): {barrio}
- Tipo de cocina: {cocina}
- Clasificación: {epigrafe}
- Tiene web propia: {tiene_web}

Devuelve ÚNICAMENTE un objeto JSON válido, sin texto adicional ni markdown, con esta estructura exacta:
{{
  "posicionamiento_precio": "uno de: low_cost, medio, premium, alta_gama",
  "tipo_clientela": "uno de: turista, profesional, residente, mixta",
  "ambiente": "uno de: moderno, tradicional, casual, formal, familiar",
  "presencia_digital": "uno de: nula, basica, activa, sofisticada",
  "resumen": "una o dos frases describiendo el perfil comercial del restaurante"
}}"""
    return prompt

# Probar el prompt con el primer restaurante
prompt_ejemplo = construir_prompt(muestra.iloc[0])
print(prompt_ejemplo)

Eres un analista de mercado especializado en el sector de restauración en Madrid.

A partir de la información disponible de un restaurante, infiere sus características comerciales. Basa tu inferencia en el nombre, el tipo de cocina y el barrio. No inventes datos factuales concretos (teléfonos, premios, fechas); limítate a caracterizar el perfil comercial.

DATOS DEL RESTAURANTE:
- Nombre: La Casa del Abuelo
- Barrio (distrito Centro de Madrid): SOL
- Tipo de cocina: regional
- Clasificación: BAR RESTAURANTE
- Tiene web propia: sí

Devuelve ÚNICAMENTE un objeto JSON válido, sin texto adicional ni markdown, con esta estructura exacta:
{
  "posicionamiento_precio": "uno de: low_cost, medio, premium, alta_gama",
  "tipo_clientela": "uno de: turista, profesional, residente, mixta",
  "ambiente": "uno de: moderno, tradicional, casual, formal, familiar",
  "presencia_digital": "uno de: nula, basica, activa, sofisticada",
  "resumen": "una o dos frases describiendo el perfil comercial del rest

In [12]:
# =====================================
# PROBAR ENRIQUECIMIENTO CON LA MUESTRA
# =====================================

def enriquecer_restaurante(restaurante):
    """Envía un restaurante a Gemini y devuelve el JSON parseado."""
    prompt = construir_prompt(restaurante)
    try:
        respuesta = client.models.generate_content(
            model=MODELO,
            contents=prompt
        )
        texto = respuesta.text.strip()
        # Limpiar posibles marcas de markdown (```json ... ```)
        texto = texto.replace("```json", "").replace("```", "").strip()
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"error": "respuesta no es JSON válido", "raw": texto[:200]}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

# Probar con los 5 de la muestra
print("=== PRUEBA DE ENRIQUECIMIENTO ===\n")
for idx, row in muestra.iterrows():
    print(f"🍽️  {row['nombre']} ({row['cocina']}, {row['barrio'].strip()})")
    resultado = enriquecer_restaurante(row)
    for clave, valor in resultado.items():
        print(f"    {clave}: {valor}")
    print()
    time.sleep(2)  # pausa para no saturar la API

=== PRUEBA DE ENRIQUECIMIENTO ===

🍽️  La Casa del Abuelo (regional, SOL)


    posicionamiento_precio: medio
    tipo_clientela: turista
    ambiente: tradicional
    presencia_digital: activa
    resumen: Establecimiento tradicional de cocina regional ubicado en un enclave estratégico de alto tránsito en el centro de Madrid. Su perfil comercial se enfoca en captar al público visitante y turístico mediante una propuesta castiza respaldada por canal digital propio.

🍽️  Barinka (peruvian, SOL)
    error: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

🍽️  Taberna Griega (greek, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: activa
    resumen: Restaurante de cocina griega informal ubicado en el dinámico barrio de Universidad (Malasaña), orientado a un público variado que incluye residentes, estudiantes y visitantes. Ofrece una propuesta gastronómica mediterránea tradicional y accesible, respaldada por su propia plataforma web.

🍽️  Maricastaña (international, UNIVERSIDAD)
    error: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

🍽️  La Prensa (regional, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: tradicional
    presencia_digital: basica
    resumen: Establecimiento de corte tradicional enfocado en la cocina regional, que atiende tanto a residentes del barrio de Universidad como a visitantes y trabajadores del distrito Centro. Su modelo comercial responde al clásico bar-restaurante de proximidad, apoyado principalmente en el tráfico peatonal directo dada su escasa presencia digital.



In [13]:
# =====================================
# FUNCIÓN ROBUSTA CON REINTENTOS Y CACHE
# =====================================

# Directorio de cache: cada restaurante enriquecido se guarda como fichero JSON individual
# Así si el proceso se corta, retomamos desde donde estábamos sin rehacer nada
CACHE_DIR = PROCESSED_DIR / "cache_llm"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def enriquecer_con_reintentos(restaurante, max_reintentos=3, espera_base=5):
    """
    Enriquece un restaurante con Gemini.
    - Reintenta hasta max_reintentos veces si hay error 503 o de red.
    - Usa espera exponencial: 5s, 10s, 20s...
    - Cachea el resultado en disco para no repetir llamadas.
    """
    restaurante_id = restaurante['restaurante_id']
    cache_path = CACHE_DIR / f"{restaurante_id}.json"

    # Si ya está cacheado, devolvemos el resultado guardado
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)

    prompt = construir_prompt(restaurante)

    for intento in range(max_reintentos):
        try:
            respuesta = client.models.generate_content(
                model=MODELO,
                contents=prompt
            )
            texto = respuesta.text.strip()
            texto = texto.replace("```json", "").replace("```", "").strip()
            resultado = json.loads(texto)

            # Guardar en cache
            with open(cache_path, 'w') as f:
                json.dump(resultado, f, ensure_ascii=False, indent=2)
            return resultado

        except json.JSONDecodeError:
            # No merece la pena reintentar si el JSON está mal
            return {"error": "json_invalido", "raw": texto[:200]}

        except Exception as e:
            error_str = str(e)
            # Si es 503 o error temporal, esperamos y reintentamos
            if "503" in error_str or "UNAVAILABLE" in error_str or "429" in error_str:
                if intento < max_reintentos - 1:
                    espera = espera_base * (2 ** intento)
                    print(f"    ⏳ Error temporal, esperando {espera}s (intento {intento+1}/{max_reintentos})")
                    time.sleep(espera)
                    continue
            # Otros errores o último intento
            return {"error": f"{type(e).__name__}: {error_str[:200]}"}

    return {"error": "max_reintentos_alcanzado"}

print("✅ Función de enriquecimiento robusta lista")
print(f"Cache en: {CACHE_DIR}")

✅ Función de enriquecimiento robusta lista
Cache en: ../data/processed/cache_llm


In [14]:
# =====================================
# REEJECUTAR MUESTRA CON FUNCIÓN ROBUSTA
# =====================================

print("=== ENRIQUECIMIENTO CON REINTENTOS Y CACHE ===\n")

for idx, row in muestra.iterrows():
    print(f"🍽️  {row['nombre']} ({row['cocina']}, {row['barrio'].strip()})")
    resultado = enriquecer_con_reintentos(row)
    for clave, valor in resultado.items():
        print(f"    {clave}: {valor}")
    print()
    time.sleep(1)

# Ver cuántos ficheros hay en cache
print(f"\n📦 Total en cache: {len(list(CACHE_DIR.glob('*.json')))} restaurantes")

=== ENRIQUECIMIENTO CON REINTENTOS Y CACHE ===

🍽️  La Casa del Abuelo (regional, SOL)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: tradicional
    presencia_digital: activa
    resumen: Establecimiento emblemático enfocado en el tapeo y la gastronomía regional dentro de una zona de altísimo tránsito peatonal en pleno centro de Madrid. Su perfil comercial combina el atractivo histórico para el turismo con el consumo casual de público local en un ambiente castizo.

🍽️  Barinka (peruvian, SOL)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: basica
    resumen: Propuesta de cocina peruana de perfil casual y precio medio, orientada a captar el alto flujo peatonal del barrio de Sol. Su estrategia comercial se apoya en la popularidad de la gastronomía latina y la rotación de clientes en el centro, prescindiendo de infraestructura web propia.

🍽️  Taberna Griega (greek, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: activa
    resumen: Restaurante de cocina griega accesible ubicado en el céntrico y dinámico barrio de Universidad, enfocado a una clientela variada de residentes, jóvenes y visitantes. Su formato de taberna tradicional se complementa con un canal digital propio para captar demanda en la zona.

🍽️  Maricastaña (international, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: activa
    resumen: Establecimiento de corte cosmopolita situado en el dinámico barrio de Universidad, con un formato híbrido de bar y restaurante adaptado a diferentes momentos del día. Su propuesta internacional e informal atrae a un público diverso compuesto por residentes locales, jóvenes profesionales y turistas.

🍽️  La Prensa (regional, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: tradicional
    presencia_digital: nula
    resumen: Bar-restaurante de corte tradicional centrado en cocina regional y propuesta accesible, orientado a captar tanto a vecinos como a transeúntes en el céntrico barrio de Universidad. Su modelo de negocio se apoya en el tráfico peatonal y la fidelización local, operando sin infraestructura digital propia.


📦 Total en cache: 5 restaurantes
